# 04 Network Metrics

## Objective
Calculate graph centrality metrics for stations in the main London Underground network.

## Inputs
- `data/processed/stations_clean.csv`
- `data/processed/connections_clean.csv`

## Outputs
- `data/processed/station_metrics.csv`

## Metrics calculated
- Degree centrality
- Betweenness centrality
- Closeness centrality

## Why this step matters
These metrics identify structurally important stations and provide the first layer of criticality analysis.

In [1]:
from pathlib import Path
import pandas as pd
import networkx as nx

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

stations = pd.read_csv(PROCESSED_DIR / "stations_clean.csv")
connections = pd.read_csv(PROCESSED_DIR / "connections_clean.csv")

## Step 1: Rebuild the main graph

The cleaned graph is rebuilt and restricted to the largest connected component before centrality analysis.

In [3]:
used_station_names = set(connections["from_name"]).union(set(connections["to_name"]))
stations_filtered = stations[stations["station_name"].isin(used_station_names)].copy()

G = nx.Graph()

for _, row in stations_filtered.iterrows():
    G.add_node(
        row["station_name"],
        station_id=row["station_id"],
        lat=row["lat"],
        lon=row["lon"]
    )

for _, row in connections.iterrows():
    G.add_edge(row["from_name"], row["to_name"], line=row["line"])

largest_component_nodes = max(nx.connected_components(G), key=len)
G_main = G.subgraph(largest_component_nodes).copy()

print("Main graph nodes:", G_main.number_of_nodes())
print("Main graph edges:", G_main.number_of_edges())
print("Connected components:", nx.number_connected_components(G_main))

Main graph nodes: 272
Main graph edges: 314
Connected components: 1


In [4]:
degree_centrality = nx.degree_centrality(G_main)
betweenness_centrality = nx.betweenness_centrality(G_main)
closeness_centrality = nx.closeness_centrality(G_main)

In [5]:
metrics = pd.DataFrame({
    "station": list(G_main.nodes()),
    "degree_centrality": [degree_centrality[s] for s in G_main.nodes()],
    "betweenness_centrality": [betweenness_centrality[s] for s in G_main.nodes()],
    "closeness_centrality": [closeness_centrality[s] for s in G_main.nodes()]
})

metrics = metrics.sort_values("betweenness_centrality", ascending=False)
metrics.head(15)

,station,degree_centrality,betweenness_centrality,closeness_centrality
27,baker street,0.02583,0.354381,0.114394
74,green park,0.02214,0.341879,0.118186
266,waterloo,0.02214,0.282364,0.108792
106,liverpool street,0.01845,0.272424,0.100259
203,westminster,0.01476,0.258455,0.112635
213,bank,0.01845,0.252756,0.105284
23,bond street,0.01476,0.243395,0.116010
20,bethnal green,0.00738,0.233617,0.093674
111,mile end,0.01476,0.233096,0.087958
257,stratford,0.01107,0.222199,0.082723


## Step 2: Save station-level network metrics

The resulting centrality scores are saved for use in risk scoring and dashboard presentation.

In [6]:
metrics.to_csv(PROCESSED_DIR / "station_metrics.csv", index=False)
print("Saved corrected station_metrics.csv")

Saved corrected station_metrics.csv
